In [ ]:
user = 'magnu'

In [ ]:
import os
import pandas as pd
dir_path = fr"C:\Users\{user}\OneDrive - University of Copenhagen\Project Lanbarcode batch upload\lanebarcode"
file_names = os.listdir(dir_path)
lane_barcodes = pd.DataFrame(file_names)
helpr_path = fr"C:\Users\{user}\OneDrive - University of Copenhagen\Project Lanbarcode batch upload\xp\Copy of list_all_SeqRuns_2024-2025_edit251113JBT_XPruns.xlsx"
pool_tags = pd.read_excel(helpr_path)

special = fr"C:\Users\{user}\OneDrive - University of Copenhagen\Project Lanbarcode batch upload\xp\Copy of list_all_SeqRuns_2024-2025_edit251113JBT_twotubesPerFC.xlsx"
special_overview = pd.read_excel(special, sheet_name='Sheet1')
special_data = pd.read_excel(special, sheet_name='Sheet2')

In [ ]:
lane_barcodes.rename(columns={0: 'filename'}, inplace=True)

In [ ]:
for row in lane_barcodes.iterrows():
    split = row[1]['filename'].split('.')
    if split[1:] != ['lanebarcode', 'html']:
        raise ValueError(f"Unexpected file format: {row[1]['filename']}")
    else :

        lane_barcodes.at[row[0], 'run'] = split[0]

In [ ]:
pattern = r'^(\d{8}|\d{6})_A00706_\d{4}_(A|B)[A-Z0-9]{9}(_.*\.lanebarcode\.html|\.lanebarcode\.html)$'
mask = lane_barcodes.filename.str.match(pattern)
assert mask.all(), "Some filenames do not match the expected pattern."

# Because we have confirmed the filename structure above we can confidently parse it here: 
lane_barcodes['date'] = lane_barcodes['filename'].apply(lambda x: x.split('_')[0])
lane_barcodes['machine'] = lane_barcodes['filename'].apply(lambda x: x.split('_')[1])
lane_barcodes['run_no'] = lane_barcodes['filename'].apply(lambda x: x.split('_')[2])
lane_barcodes['full_flowcell_id'] = lane_barcodes['filename'].apply(lambda x: x.split('_')[3][:10])
lane_barcodes['side'] = lane_barcodes['full_flowcell_id'].apply(lambda x: x[0])
lane_barcodes['flowcell_id'] = lane_barcodes['full_flowcell_id'].apply(lambda x: x[1:])

In [ ]:
assert all(lane_barcodes['full_flowcell_id'].apply(len) == 10)
assert all(lane_barcodes['side'].apply(len) == 1)
assert all(lane_barcodes['flowcell_id'].apply(len) == 9)
assert all(lane_barcodes['run_no'].apply(len) == 4)
assert all(lane_barcodes['machine'].apply(len) == 6)

In [ ]:
mask1 = lane_barcodes.flowcell_id.isin(pool_tags['FlowCellID'])
mask2 = lane_barcodes.flowcell_id.isin(special_overview['FlowCellID'])
lane_barcodes = lane_barcodes[mask1]
to_upload_special = lane_barcodes[mask2]

In [ ]:
mask = lane_barcodes.flowcell_id.duplicated(keep=False)
lane_barcodes[mask]

In [ ]:
# remove bad duplicate
mask = lane_barcodes.filename == '20250117_A00706_0911_BH5F52DSX7_new.lanebarcode.html'
lane_barcodes = lane_barcodes[~mask]

In [ ]:
# remove BH5F52DSX7 entirely as it gives errors in the parsing due to only 1 lane in file
mask = lane_barcodes['flowcell_id'] == 'H5F52DSX7' 
lane_barcodes = lane_barcodes[~mask]        

Check whats already uploaded

In [ ]:
from pathlib import Path
import sys
import os

def find_project_root():
    path = Path(os.getcwd()).resolve()
    while path != path.root:
        if (path / 'very_rootsy_file.txt').exists():
            return path
        path = path.parent
    return None  # Project root not found

project_root = find_project_root()

sys.path.append(str(project_root))

In [ ]:
%env PGPASSWORD=5J8DhII0RRsPW1

In [ ]:
from constants.db_connections import ENGINE, ENGINE_READ_ONLY, SQL_ALCH_CONFIG, PSY_CONN
q = 'select * from flowcell'
flowcell_db = pd.read_sql(q, ENGINE_READ_ONLY)
flowcell_db['full_flowcell_id'] = flowcell_db['flowcell_position'] + flowcell_db['flowcell_id']

q = 'select * from top_unknown_seq_barcodes'
tusb_db = pd.read_sql(q, ENGINE_READ_ONLY)

In [ ]:
# Filter out any flowcell IDs already in the database
mask = pool_tags['FlowCellID'].isin(flowcell_db.flowcell_id)
pool_tags = pool_tags[~mask]
mask = lane_barcodes['flowcell_id'].isin(pool_tags['FlowCellID'])
lane_barcodes = lane_barcodes[mask]

Get all flowcell data to upload into one df

In [ ]:
from constants.db_names.names import data
import lane_barcode_parser

fc = []
tpb = []

for file_name in lane_barcodes.filename:
    file_path = os.path.join(dir_path, file_name)

    tube_tag_col_name = data.flowcell.library_pool_tag()
    read_len_col_name = data.flowcell.read_length()
    seq_machine_col_name = data.flowcell.sequencing_machine()
    seq_date_col_name = data.flowcell.sequencing_date()
    flowcell_pos_col_name = data.flowcell.flowcell_position()
    seq_run_col_name = data.flowcell.sequencing_run()
    print(file_name)
    print(file_path)
    sheet = pd.read_html(file_path, thousands=',', decimal='.')

    flowcell_data, top_unknown_barcodes = lane_barcode_parser.parse(df=sheet)
    flowcell_data['upload_sheet'] = str(file_name)
    top_unknown_barcodes['upload_sheet'] = str(file_name)

    fc.append(flowcell_data)
    tpb.append(top_unknown_barcodes)

In [ ]:
top_unknown_barcodes = pd.concat(tpb, ignore_index=True)
flowcell_data = pd.concat(fc, ignore_index=True)

In [ ]:
pool_tags.rename(columns={'FlowCellID': 'Flowcell ID'}, inplace=True)

In [ ]:
pool_tags['key'] = pool_tags['Flowcell ID'].astype(str) + '_' + pool_tags['Lane'].astype(str)
flowcell_data['key'] = flowcell_data['Flowcell ID'].astype(str) + '_' + flowcell_data['Lane'].astype(str)
top_unknown_barcodes['key'] = top_unknown_barcodes['Flowcell ID'].astype(str) + '_' + top_unknown_barcodes['Lane'].astype(str)

In [ ]:
flowcell_data = flowcell_data.merge(pool_tags, on='key', how='inner', validate='many_to_one')

In [ ]:
top_unknown_barcodes = top_unknown_barcodes.merge(pool_tags, on='key', how='inner', validate='many_to_one')

Get read length, sequencing date, tubetag, seq machine, seq run and flowcell pos merged in

In [ ]:
machine_id_map = {
                'A00706': {
                    'name': 'Nova Seq 6000',
                    'rl': '101'
                }
            }
top_unknown_barcodes.loc[:, 'read_length'] = machine_id_map['A00706']['rl']
top_unknown_barcodes.loc[:, 'sequencing_machine'] = machine_id_map['A00706']['name']
flowcell_data.loc[:, 'read_length'] = machine_id_map['A00706']['rl']
flowcell_data.loc[:, 'sequencing_machine'] = machine_id_map['A00706']['name']

In [ ]:
q = 'select column_name_db, column_name_sheet from name_maps.column_names where table_id = 9'
flowcell_name_maps = pd.read_sql(q, ENGINE_READ_ONLY)
flowcell_name_maps = flowcell_name_maps.dropna()
flowcell_name_maps.index = flowcell_name_maps['column_name_sheet']
flowcell_name_maps.drop(columns=['column_name_sheet'], inplace=True)

In [ ]:
q = 'select column_name_db, column_name_sheet from name_maps.column_names where table_id = 18'
tusb_name_maps = pd.read_sql(q, ENGINE_READ_ONLY)
tusb_name_maps = tusb_name_maps.dropna()
tusb_name_maps.index = tusb_name_maps['column_name_sheet']
tusb_name_maps.drop(columns=['column_name_sheet'], inplace=True)

In [ ]:
translator_tusb = tusb_name_maps.to_dict(orient='dict')['column_name_db']
translator_fc = flowcell_name_maps.to_dict(orient='dict')['column_name_db']

In [ ]:
flowcell_data = flowcell_data.rename(columns=translator_fc)
top_unknown_barcodes = top_unknown_barcodes.rename(columns=translator_tusb)

In [ ]:
top_unknown_barcodes.drop(columns=['key'], inplace=True)
top_unknown_barcodes.drop(columns=['Flowcell ID_x'], inplace=True)
top_unknown_barcodes.rename(columns={'Flowcell ID_y': 'Flowcell ID'}, inplace=True)
top_unknown_barcodes.drop(columns=['Lane_x'], inplace=True)
top_unknown_barcodes.rename(columns={'Lane_y': 'Lane'}, inplace=True)
flowcell_data.drop(columns=['key'], inplace=True)
flowcell_data.drop(columns=['Flowcell ID_x'], inplace=True)
flowcell_data.rename(columns={'Flowcell ID_y': 'Flowcell ID'}, inplace=True)
flowcell_data.drop(columns=['Lane_x'], inplace=True)
flowcell_data.rename(columns={'Lane_y': 'Lane'}, inplace=True)

In [ ]:
fc_to_upload = flowcell_data   
tusb_to_upload = top_unknown_barcodes

tusb_to_upload['database_insert_by'] = 'julie.bitz-thorsen@sund.ku.dk'
fc_to_upload['database_insert_by'] = 'julie.bitz-thorsen@sund.ku.dk'
from datetime import datetime, timezone

ts = datetime.now(timezone.utc)  
tusb_to_upload['database_insert_datetime_utc'] = ts
fc_to_upload['database_insert_datetime_utc'] = ts
from uuid import uuid4
tusb_to_upload['upload_uuid'] = uuid4()
tusb_to_upload = tusb_to_upload.replace(["nan", "NaN", "NAN", "null", "NULL", "None", "none"], None)
fc_to_upload['upload_uuid'] = uuid4()
fc_to_upload = fc_to_upload.replace(["nan", "NaN", "NAN", "null", "NULL", "None", "none"], None)

In [ ]:
tusb_to_upload.drop(columns=['ID', 
                             'BSSHRunID',
                             'WorkingID',
                             'RunSheetID',
                             'InstrumentID',
                             'Notes by Julie',
                             'SequencingOK',
                             'full_flowcell_id',
                             'duplicate_flowcell',
                             'run',
                             'date',
                             'machine',
                             'side',
                             'flowcell', 
                             'ttags'],
                             inplace=True, errors='ignore')
fc_to_upload.drop(columns=['ID', 
                             'BSSHRunID',
                             'WorkingID',
                             'RunSheetID',
                             'InstrumentID',
                             'Notes by Julie',
                             'SequencingOK',
                             'full_flowcell_id',
                             'duplicate_flowcell',
                             'run',
                             'date',
                             'machine',
                             'side',
                             'flowcell', 
                             'ttags'],
                             inplace=True, errors='ignore')


In [ ]:
tusb_to_upload = tusb_to_upload.rename(columns=translator_tusb)
fc_to_upload = fc_to_upload.rename(columns=translator_fc)

In [ ]:
renamer = {
    'RunDate': 'sequencing_date',
    'Tubetag': 'seqc_tube_tag',
    'InstrumentSide': 'flowcell_position'
}

fc_to_upload.rename(columns=renamer, inplace=True)



In [ ]:
red_cols = set(tusb_to_upload.columns) - set(tusb_db.columns)
print(red_cols)
tusb_to_upload.drop(columns=red_cols, inplace=True)
red_cols = set(fc_to_upload.columns) - set(flowcell_db.columns)
print(red_cols)
fc_to_upload.drop(columns=red_cols, inplace=True)

In [ ]:
fc_to_upload.to_sql('flowcell', ENGINE, if_exists='append', index=False)


In [ ]:
tusb_to_upload